# 02. Download and construct Stage 2 pre-pandemic predictors (September 2018 OMB)

Builds one CSV row per eligible CBSA and *only uses indicators first publicly released by February 29, 2020*:

- **ACS 2018 five-year (2014–2018)** estimates, released December 19, 2019.
- **QCEW 2019 Q2**, released December 4, 2019, and QCEW 2018 Q2.
- September 2018 OMB county-to-CBSA delineations inherited from notebook 00.

No QCEW observations after June 2019 enter predictors. Later-revised historical downloads may differ from the original releases; the manifest explicitly notes this limitation.

**Full-run configuration:** All panel-eligible CBSAs and all 19 private-sector shares are retained. If the cached ACS 2018 file is absent, provide the Census API key securely in the kernel before the ACS download cell.


Revised predictor acquisition adds ACS labor-force participation, grouped income distribution median, broad occupations, 2018 LAUS, 2015--2018 annual growth, CBSA type, complete-sector diversification, exact-geography May 2018 OES, and an explicitly labeled archived-O*NET task-feasibility proxy. No 2019 ACS or post-cutoff published telework score is used. See RERUN_DESIGN_2020.md.


The primary feature set uses broad private-industry exposure, ACS occupational composition and household broadband. Detailed industry exposures and the OES task proxy are exported with their original coverage limits for Step 05.


In [11]:
from pathlib import Path
from datetime import datetime,timezone
import os,zipfile,json,hashlib
import numpy as np,pandas as pd,requests
ROOT=Path(os.environ['RL_PROJECT_ROOT']).resolve() if os.environ.get('RL_PROJECT_ROOT') else next(
 (p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data'/'processed'/'omb2018_geography_manifest.json').exists()),None)
if ROOT is None:raise FileNotFoundError('Run notebook 00 first')
RAW=ROOT/'data'/'raw';PROC=ROOT/'data'/'processed';OUT=ROOT/'output'
for p in (RAW/'acs',RAW/'qcew',PROC,OUT):p.mkdir(parents=True,exist_ok=True)
manifest=json.loads((PROC/'omb2018_geography_manifest.json').read_text())
if manifest['geography_vintage']!='2018-09':raise ValueError('Wrong OMB vintage')
CW=PROC/'county_cbsa_crosswalk_sep2018.csv'
PANEL=PROC/'qcew_cbsa_monthly_2010_2023_omb2018.csv'
cw=pd.read_csv(CW,dtype={'cbsa_code':'string','county_fips':'string','state_fips':'string'})
panel=pd.read_csv(PANEL,dtype={'cbsa_code':'string'},parse_dates=['date'])
cw['cbsa_code']=cw.cbsa_code.str.zfill(5);cw['county_fips']=cw.county_fips.str.zfill(5)
panel['cbsa_code']=panel.cbsa_code.str.zfill(5)
ids=set(panel.cbsa_code.unique())
geo=cw.loc[cw.cbsa_code.isin(ids),['cbsa_code','county_fips','state_fips']].copy()
assert not geo.duplicated('county_fips').any()

def sha256(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for x in iter(lambda:f.read(4*1024*1024),b''):h.update(x)
 return h.hexdigest()
if sha256(CW)!=manifest['crosswalk_sha256'] or sha256(PANEL)!=manifest['panel_sha256']:
 raise ValueError('Geography source changed since notebook 00')

def fetch_once(url,path,kind='csv'):
 if path.exists():
  if path.stat().st_size==0:raise ValueError(f'Empty cached file: {path}')
  if kind=='zip' and not zipfile.is_zipfile(path):raise ValueError(f'Corrupted cached ZIP: {path}')
  print('REUSE:',path.relative_to(ROOT));return path
 tmp=path.with_suffix(path.suffix+'.part')
 with requests.get(url,stream=True,timeout=(30,180),headers={'User-Agent':'Academic replication project'}) as r:
  r.raise_for_status()
  with tmp.open('wb') as f:
   for chunk in r.iter_content(1024*1024):
    if chunk:f.write(chunk)
 if not tmp.exists() or not tmp.stat().st_size or (kind=='zip' and not zipfile.is_zipfile(tmp)):
  raise ValueError('Invalid downloaded data: '+url)
 tmp.replace(path);return path
print('Eligible 2018 geography CBSAs:',len(ids))

Eligible 2018 geography CBSAs: 936


## 1. Download 2018 ACS five-year county estimates, only when missing

In [12]:
ACS_VARS=['NAME','B01003_001E','B15003_001E','B15003_022E','B15003_023E',
          'B15003_024E','B15003_025E','B17001_001E','B17001_002E',
          'B23025_003E','B23025_005E','B28002_001E','B28002_004E']
ACS_URL='https://api.census.gov/data/2018/acs/acs5'
ACS_FILE=RAW/'acs'/'acs5_2018_counties.csv'

# The Census API can return a non-JSON body even when a request completes.
# The Census 2018 API documentation now requests an API key. Keep it in an
# environment variable, not in Git or your notebook source.
# In a notebook cell, if needed: import getpass,os; os.environ['CENSUS_API_KEY']=getpass.getpass('Census API key: ')
def fetch_acs2018():
    import requests
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry

    params={'get':','.join(ACS_VARS),'for':'county:*','in':'state:*'}
    key=os.getenv('CENSUS_API_KEY','').strip()
    if not key:
        raise RuntimeError(
            'The Census Bureau now requires an API key for all Census Data API queries (since 2026). '
            'Request a free key at https://api.census.gov/data/key_signup.html, '
            'then set CENSUS_API_KEY in the Jupyter kernel using getpass before rerunning this cell. '
            'If you already have a valid cached ACS CSV, this download step is skipped.'
        )
    params['key']=key

    retries=Retry(total=3,connect=3,read=3,backoff_factor=2,
                  status_forcelist=(429,500,502,503,504),
                  allowed_methods=frozenset(['GET']))
    with requests.Session() as session:
        session.mount('https://',HTTPAdapter(max_retries=retries))
        try:
            response=session.get(ACS_URL,params=params,timeout=(30,150),
                                 headers={'Accept':'application/json',
                                          'User-Agent':'Research-replication/1.0'})
            response.raise_for_status()
        except requests.RequestException as exc:
            status=getattr(getattr(exc,'response',None),'status_code',None)
            detail=getattr(getattr(exc,'response',None),'text','')[:600].replace(key,'[REDACTED]')
            raise RuntimeError(
                'Census 2018 ACS API HTTP/network error. '
                f'Status={status}; response excerpt={detail!r}. '
                'Check your connection and CENSUS_API_KEY, then rerun.'
            ) from None

        try:
            payload=response.json()
        except ValueError as exc:
            excerpt=response.text[:800].replace('\n',' ').replace(key,'[REDACTED]')
            raise RuntimeError(
                'The 2018 ACS Census API returned non-JSON content, so the CSV '
                'was NOT created. '
                f'HTTP {response.status_code}; Content-Type={response.headers.get("Content-Type", "unknown")}; '
                f'body begins: {excerpt!r}. '
                'Census API documentation now requests an API key. '
                'Set CENSUS_API_KEY in your environment if the body indicates '
                'authorization/rate limits, then rerun this cell.'
            ) from None

    if (not isinstance(payload,list) or len(payload)<2 or
            not isinstance(payload[0],list)):
        raise RuntimeError('Census API returned an unexpected JSON response structure; raw response withheld to protect credentials.')
    frame=pd.DataFrame(payload[1:],columns=payload[0])
    required=set(ACS_VARS)-{'NAME'}
    missing=required.union({'state','county'}).difference(frame.columns)
    if missing:
        raise RuntimeError(f'Census API response is missing columns: {sorted(missing)}')
    if frame.empty or frame.duplicated(['state','county']).any():
        raise RuntimeError('Census API response is empty or has duplicate county rows.')
    ACS_FILE.parent.mkdir(parents=True,exist_ok=True)
    # Atomic caching: only cache a valid table. Never save HTML/error text as CSV.
    tmp=ACS_FILE.with_suffix('.csv.part')
    frame.to_csv(tmp,index=False)
    tmp.replace(ACS_FILE)
    (RAW/'acs'/'acs5_2018_request.json').write_text(json.dumps({
        'retrieved_utc':datetime.now(timezone.utc).isoformat(),
        'source_url':ACS_URL,
        'query_variables':ACS_VARS,
        'geography':'county:* in state:*',
        'first_released':'2019-12-19',
        'uses_api_key':bool(key),
        'note':'Key itself deliberately omitted from public metadata.'
    },indent=2))
    return frame

if ACS_FILE.exists():
    if ACS_FILE.stat().st_size==0:
        raise ValueError(f'Cached ACS CSV is empty: {ACS_FILE}; remove invalid cache and rerun.')
    acs=pd.read_csv(ACS_FILE,dtype='string')
    print('REUSE:',ACS_FILE.relative_to(ROOT))
else:
    acs=fetch_acs2018()
    print('DOWNLOADED:',ACS_FILE.relative_to(ROOT),'| counties:',len(acs))

acs['county_fips']=acs['state'].astype('string').str.zfill(2)+acs['county'].astype('string').str.zfill(3)
for v in ACS_VARS:
 if v!='NAME':
  acs[v]=pd.to_numeric(acs[v],errors='coerce')
  acs.loc[acs[v]<0,v]=np.nan
ac=geo.merge(acs[['county_fips']+[x for x in ACS_VARS if x!='NAME']],on='county_fips',how='left',validate='one_to_one')
varnames=[x for x in ACS_VARS if x!='NAME']
complete=ac[varnames].notna().all(axis=1)
coverage=ac.assign(valid=complete).groupby('cbsa_code').valid.agg(['sum','size'])
cbsa_acs=ac.groupby('cbsa_code')[varnames].sum(min_count=1)
cbsa_acs.loc[coverage['sum'].ne(coverage['size']),:]=np.nan

def ratio(n,d):return n.div(d.where(d>0))
a=pd.DataFrame(index=cbsa_acs.index)
a['acs2018_population']=cbsa_acs.B01003_001E
a['acs2018_bachelors_plus_share']=ratio(
 cbsa_acs[['B15003_022E','B15003_023E','B15003_024E','B15003_025E']].sum(axis=1,min_count=4),
 cbsa_acs.B15003_001E)
a['acs2018_poverty_share']=ratio(cbsa_acs.B17001_002E,cbsa_acs.B17001_001E)
a['acs2018_unemployment_share']=ratio(cbsa_acs.B23025_005E,cbsa_acs.B23025_003E)
a['acs2018_broadband_subscription_share']=ratio(cbsa_acs.B28002_004E,cbsa_acs.B28002_001E)
a.to_csv(PROC/'stage2_omb2018_acs2018_predictors.csv')
print('Complete ACS CBSAs:',int(coverage['sum'].eq(coverage['size']).sum()))

REUSE: data\raw\acs\acs5_2018_counties.csv
Complete ACS CBSAs: 935


## 2. Use 2018 Q2 and 2019 Q2 QCEW employment from cached archives


In [13]:
QCEW_URL='https://data.bls.gov/cew/data/files/{year}/csv/{year}_qtrly_by_area.zip'
PRIVATE_SECTORS=['11','21','22','23','31-33','42','44-45','48-49','51','52',
                 '53','54','55','56','61','62','71','72','81']
BROAD_PRIVATE_SECTORS=['101','102']  # goods-producing, service-providing
NEED=['area_fips','own_code','industry_code','year','qtr','disclosure_code',
      'month1_emplvl','month2_emplvl','month3_emplvl']

def read_q2(year,industry_codes,ownerships):
    """Retrieve requested ownership/industry cells for Q2 from cached raw QCEW ZIP."""
    path=fetch_once(QCEW_URL.format(year=year),
                    RAW/'qcew'/f'{year}_qtrly_by_area.zip',kind='zip')
    pieces=[]
    with zipfile.ZipFile(path) as archive:
        for filename in archive.namelist():
            if not filename.lower().endswith('.csv'):
                continue
            with archive.open(filename) as f:
                for chunk in pd.read_csv(f,dtype=str,usecols=lambda x:x in NEED,
                                         chunksize=100000,low_memory=False,encoding='latin1'):
                    if not set(NEED).issubset(chunk.columns):
                        raise ValueError(f'Missing QCEW columns in {filename}')
                    for col in ['area_fips','own_code','industry_code','qtr','year','disclosure_code']:
                        chunk[col]=chunk[col].astype('string').str.strip()
                    valid=(chunk.area_fips.str.fullmatch(r'\d{5}',na=False)
                           & ~chunk.area_fips.str.endswith('000',na=False)
                           & chunk.own_code.isin(ownerships)
                           & chunk.year.eq(str(year)) & chunk.qtr.eq('2')
                           & chunk.industry_code.isin(industry_codes))
                    if valid.any():
                        pieces.append(chunk.loc[valid].copy())
    if not pieces:
        raise ValueError(f'No Q2 county industry data for {year}')
    q=pd.concat(pieces,ignore_index=True)
    keys=['area_fips','own_code','industry_code']
    if q.duplicated(keys).any():
        example=q.loc[q.duplicated(keys,keep=False),keys].head().to_dict('records')
        raise ValueError(f'Duplicate 2019 Q2 QCEW county/ownership/industry records: {example}')
    for m in (1,2,3):
        col=f'month{m}_emplvl'
        q[col]=pd.to_numeric(q[col],errors='coerce')
    month_cols=[f'month{i}_emplvl' for i in (1,2,3)]
    q['average_employment']=q[month_cols].mean(axis=1)
    suppressed=q.disclosure_code.fillna('').str.upper().eq('N')
    incomplete=q[month_cols].isna().any(axis=1)
    q.loc[suppressed | incomplete,'average_employment']=np.nan
    q=q.rename(columns={'area_fips':'county_fips'})
    return q[['county_fips','own_code','industry_code','average_employment','disclosure_code']]

# Do not change the two source periods; both Q2 years were public by Feb 2020.
q18=read_q2(2018,industry_codes=['10'],ownerships=['0'])
q19=read_q2(2019,industry_codes=['10']+PRIVATE_SECTORS+BROAD_PRIVATE_SECTORS,
            ownerships=['0','1','2','3','5'])
# All ownerships are acquired for the planned covered-employment sector exposures.
assert (q19.own_code.eq('5') & q19.industry_code.isin(PRIVATE_SECTORS)).any(), (
    'Private-industry sector records missing; inspect the raw QCEW source.')
print('2019 Q2 county-level records by ownership:')
print(q19.groupby('own_code').size().to_string())


REUSE: data\raw\qcew\2018_qtrly_by_area.zip
REUSE: data\raw\qcew\2019_qtrly_by_area.zip
2019 Q2 county-level records by ownership:
own_code
0     3273
1    13500
2    14933
3    25624
5    68630


## 3. Aggregate ownership-consistent county employment and private-industry shares


In [14]:
n_expected=geo.groupby('cbsa_code').county_fips.size()

def complete_total(q,name,ownership):
    subset=q.loc[q.industry_code.eq('10') & q.own_code.eq(ownership),
                 ['county_fips','average_employment']]
    joined=geo[['cbsa_code','county_fips']].merge(
        subset,on='county_fips',how='left',validate='one_to_one')
    gr=joined.groupby('cbsa_code').average_employment
    return gr.sum(min_count=1).where(gr.count().eq(n_expected)).rename(name)

T18=complete_total(q18,'qcew2018q2_employment','0')
T19=complete_total(q19,'qcew2019q2_employment','0')
P19=complete_total(q19,'qcew2019q2_private_employment','5')
base=pd.concat([T18,T19,P19],axis=1)
base['qcew2018to2019q2_growth']=base.qcew2019q2_employment.div(
    base.qcew2018q2_employment.where(base.qcew2018q2_employment>0))-1
base['log_qcew2019q2_employment']=np.log(
    base.qcew2019q2_employment.where(base.qcew2019q2_employment>0))
base['qcew2019q2_private_share_all_covered']=P19.div(T19.where(T19>0))
# Government employment is the complement, so do not also include government share
# as a separate numeric predictor in the same regression.
invalid_private_share=base.qcew2019q2_private_share_all_covered.notna() & (
    (base.qcew2019q2_private_share_all_covered<0) |
    (base.qcew2019q2_private_share_all_covered>1.001))
if invalid_private_share.any():
    raise ValueError('Private employment exceeds total covered employment for some CBSAs. '
                     'Check ownership and county records.')

private=q19.loc[q19.own_code.eq('5') &
                q19.industry_code.isin(PRIVATE_SECTORS+BROAD_PRIVATE_SECTORS),
                ['county_fips','industry_code','average_employment','disclosure_code']]
industry_codes=PRIVATE_SECTORS+BROAD_PRIVATE_SECTORS
expected_grid=pd.MultiIndex.from_product(
    [geo.county_fips.unique(),industry_codes],
    names=['county_fips','industry_code']).to_frame(index=False)
expected_grid=expected_grid.merge(geo[['county_fips','cbsa_code']],
                                  on='county_fips',validate='many_to_one')
expected_grid=expected_grid.merge(private,on=['county_fips','industry_code'],
                                   how='left',validate='one_to_one',indicator='_merge')
expected_grid['absent_structural_zero']=expected_grid['_merge'].eq('left_only')
expected_grid['suppressed_N']=expected_grid.disclosure_code.fillna('').str.upper().eq('N')
expected_grid['invalid_reported']=~expected_grid.absent_structural_zero & ~expected_grid.suppressed_N & expected_grid.average_employment.isna()
# Structural zero is not the same as disclosure suppression; never overwrite N with 0.
expected_grid.loc[expected_grid.absent_structural_zero,'average_employment']=0.0
expected_grid['usable']=expected_grid.average_employment.notna()
status=(expected_grid.groupby(['cbsa_code','industry_code'])
        .agg(county_expected=('county_fips','size'),
             county_reported=('usable','sum'),
             n_structural_zero=('absent_structural_zero','sum'),
             n_suppressed_N=('suppressed_N','sum'),
             n_invalid_reported=('invalid_reported','sum'),
             industry_employment=('average_employment',lambda x:x.sum(min_count=1)))
        .reset_index())
status.loc[status.county_reported<status.county_expected,'industry_employment']=np.nan
status['all_counties_reported']=status.county_reported.eq(status.county_expected)
expected_grid.to_csv(OUT/'stage2_omb2018_qcew_county_sector_status.csv',index=False)
print('QCEW absent industry cells recoded to zero:',int(expected_grid.absent_structural_zero.sum()),
      '| explicitly suppressed N:',int(expected_grid.suppressed_N.sum()),
      '| invalid published rows:',int(expected_grid.invalid_reported.sum()))
status.to_csv(OUT/'stage2_omb2018_qcew_sector_coverage_detail.csv',index=False)
wide=status.pivot(index='cbsa_code',columns='industry_code',values='industry_employment')
wide=wide.reindex(index=T19.index,columns=industry_codes)

private_shares=wide[PRIVATE_SECTORS].div(P19.where(P19>0),axis=0)
private_shares.columns=['qcew2019q2_share_naics_'+s.replace('-','_')
                        for s in private_shares.columns]
# Robust broad alternative: goods-producing share (101), no 102 complement.
private_shares['qcew2019q2_private_goods_producing_share']=(
    wide['101'].div(P19.where(P19>0)))
shares=private_shares

sector_complete=wide[PRIVATE_SECTORS].notna().sum(axis=1)
audit=pd.DataFrame({
    'n_valid_sectors':sector_complete,
    'n_sector_definitions':len(PRIVATE_SECTORS),
    'private_employment_available':P19.notna(),
    'broad_goods_share_available':wide['101'].notna() & P19.notna(),
    'industry_totals_ratio':wide[PRIVATE_SECTORS].sum(
        axis=1,min_count=len(PRIVATE_SECTORS)).div(P19.where(P19>0))
})
audit.to_csv(OUT/'stage2_omb2018_qcew_industry_coverage.csv')
print('CBSAs with all 19 private-sector NAICS shares:',
      int(audit.n_valid_sectors.eq(len(PRIVATE_SECTORS)).sum()))
print('CBSAs with at least one available detailed industry share:',
      int(audit.n_valid_sectors.gt(0).sum()))
print('CBSAs with available broad goods share:',
      int(audit.broad_goods_share_available.sum()))
print('Coverage distribution:')
print(audit.n_valid_sectors.describe().to_string())


QCEW absent industry cells recoded to zero: 1103 | explicitly suppressed N: 6523 | invalid published rows: 0
CBSAs with all 19 private-sector NAICS shares: 178
CBSAs with at least one available detailed industry share: 935
CBSAs with available broad goods share: 924
Coverage distribution:
count    936.000000
mean      14.255342
std        4.290017
min        0.000000
25%       12.000000
50%       15.000000
75%       17.000000
max       19.000000


## 4. Export predictor CSV and geographic validation groups

A broad Census division for each CBSA is assigned using the division with the most Q2 2019 covered employment, using *only pre-cutoff data*. The division is a model-validation grouping variable, not a predictor. A QCEW county/industry cell absent from the public file is a **structural zero**; an observed row whose `disclosure_code == 'N'` is suppressed and remains **missing**. Invalid non-suppressed numeric cells also remain missing and are audited. Missing sector shares are imputed only inside training folds.

In [15]:
DIVISIONS={
 'New England':['09','23','25','33','44','50'],
 'Middle Atlantic':['34','36','42'],
 'East North Central':['17','18','26','39','55'],
 'West North Central':['19','20','27','29','31','38','46'],
 'South Atlantic':['10','11','12','13','24','37','45','51','54'],
 'East South Central':['01','21','28','47'],
 'West South Central':['05','22','40','48'],
 'Mountain':['04','08','16','30','32','35','49','56'],
 'Pacific':['02','06','15','41','53'],
 'Puerto Rico':['72']}
state_to_div={st:region for region,states in DIVISIONS.items() for st in states}
# Audit geographic composition of the FULL 2018 delineation, including excluded CBSAs.
# Majority by county count is for sample-selection auditing only; the actual held-out
# validation group below is assigned using Q2 2019 covered-employment weights.
audit_path=OUT/'omb2018_cbsa_sample_selection_audit.csv'
if audit_path.exists():
    selection=pd.read_csv(audit_path,dtype={'cbsa_code':'string'})
    candidate=cw[['cbsa_code','county_fips','state_fips']].copy()
    candidate['audit_division']=candidate.state_fips.map(state_to_div)
    candidate=(candidate.groupby(['cbsa_code','audit_division'],as_index=False)
               .county_fips.count().sort_values(['cbsa_code','county_fips','audit_division'],
                                                ascending=[True,False,True]))
    dominant=candidate.drop_duplicates('cbsa_code')[['cbsa_code','audit_division']]
    selection=selection.merge(dominant,on='cbsa_code',how='left',validate='one_to_one')
    selection.to_csv(OUT/'omb2018_cbsa_sample_selection_audit_with_division.csv',index=False)
    by_division=selection.groupby(['audit_division','cbsa_type','pre_cutoff_eligible',
                                  'full_history_included'],dropna=False).agg(
        n_cbsa=('cbsa_code','size'),
        median_employment_june2019=('employment_june2019','median')).reset_index()
    by_division.to_csv(OUT/'omb2018_cbsa_sample_selection_by_division.csv',index=False)
    print('Sample selection by geographic division (county-majority audit assignment):')
    print(by_division.to_string(index=False))
else:
    print('Sample selection audit missing; rerun updated notebook 00 for division audit.')
weights=geo[['cbsa_code','county_fips','state_fips']].merge(
 q19.loc[q19.industry_code.eq('10') & q19.own_code.eq('0'),['county_fips','average_employment']],
 on='county_fips',how='left',validate='one_to_one')
weights['division']=weights.state_fips.map(state_to_div)
if weights.division.isna().any():raise ValueError('Unrecognized state FIPS for division grouping')
w=weights.groupby(['cbsa_code','division'],as_index=False).average_employment.sum(min_count=1)
w=w.sort_values(['cbsa_code','average_employment','division'],ascending=[True,False,True])
# Operational principal county is the largest constituent county by pre-cutoff
# employment; ties resolved by FIPS. Do not use later outcomes for assignment.
principal=weights.sort_values(['cbsa_code','average_employment','county_fips'],ascending=[True,False,True]).drop_duplicates('cbsa_code')
principal[['cbsa_code','county_fips','division','average_employment']].to_csv(OUT/'stage2_omb2018_principal_county_groups.csv',index=False)
group=principal.set_index('cbsa_code').division.rename('geographic_group')
predictors=base.join(a,how='left').join(shares,how='left').join(group,how='left')
predictors=predictors.reindex(sorted(ids))
predictors.index.name='cbsa_code'
predictors=predictors.reset_index()
if predictors.cbsa_code.isna().any() or predictors.cbsa_code.duplicated().any():
 raise ValueError('CBSA sample duplicate or missing')
if set(predictors.cbsa_code)!=ids:raise ValueError('Predictor and 2018 employment samples differ')
import sys
if str(ROOT/'code') not in sys.path: sys.path.insert(0,str(ROOT/'code'))
from design_predictors import extend_predictors, SCHEMA
from design_models import FEATURE_DESIGN, write_coverage_audit
predictors,extended_meta=extend_predictors(ROOT,predictors,geo,shares,q19)
path=PROC/'stage2_omb2018_pre2020_predictors.csv'
predictors.to_csv(path,index=False)
write_coverage_audit(predictors, OUT/'stage2_predictor_coverage_comparison.csv')
features=predictors.select_dtypes(include='number')
coverage=pd.DataFrame({'feature':features.columns,'nonmissing':features.notna().sum().values})
coverage['total_cbsa']=len(predictors)
coverage['coverage_pct']=coverage.nonmissing/len(predictors)*100
coverage.to_csv(OUT/'stage2_omb2018_predictor_coverage.csv',index=False)
manifest2={'feature_design':FEATURE_DESIGN,'design_schema':SCHEMA,'extended_predictors_manifest_sha256':sha256(OUT/'stage2_omb2018_extended_predictors_manifest.json'),'geography_vintage':'2018-09','information_cutoff':'2020-02-29',
 'acs2018_5year_release':'2019-12-19','qcew2019q2_release':'2019-12-04',
 'qcew_quarters_used':['2018-Q2','2019-Q2'],
 'predictors_file':str(path.relative_to(ROOT)),
 'predictors_sha256':sha256(path),
 'geography_panel_sha256':sha256(PANEL),
 'qcew_industry_ownership':'5 (private-sector NAICS composition)',
  'absent_private_sector_rows_treated_as_zero':True,
  'suppressed_N_rows_treated_as_missing':True,
  'county_sector_status_sha256':sha256(OUT/'stage2_omb2018_qcew_county_sector_status.csv'),
 'qcew_total_ownership':'0 (all covered employment)',
 'industry_share_denominator':'2019 Q2 private-sector total employment',
 'industry_coverage_audits':['output/stage2_omb2018_qcew_industry_coverage.csv',
   'output/stage2_omb2018_qcew_sector_coverage_detail.csv'],
 'historical_QCEW_vintage_verified':False,
 'note':'QCEW ZIP data downloaded today may reflect post-release revisions; source observations are no later than 2019-Q2.',
 'n_cbsa':len(predictors)}
(OUT/'stage2_omb2018_predictors_manifest.json').write_text(json.dumps(manifest2,indent=2))
print('Saved:',path.relative_to(ROOT),'| CBSAs:',len(predictors))

Sample selection by geographic division (county-majority audit assignment):
    audit_division                     cbsa_type  pre_cutoff_eligible  full_history_included  n_cbsa  median_employment_june2019
East North Central Metropolitan Statistical Area                 True                   True      61                     94387.0
East North Central Micropolitan Statistical Area                 True                   True     102                     18443.5
East South Central Metropolitan Statistical Area                 True                   True      30                     87344.0
East South Central Micropolitan Statistical Area                 True                   True      66                     16118.5
   Middle Atlantic Metropolitan Statistical Area                 True                   True      35                    123574.0
   Middle Atlantic Micropolitan Statistical Area                 True                   True      31                     23285.0
          Mountain Me